In [494]:
import random
import math
import torch

class Neuron:
        
    def __init__(self, nin):
        self.w = [torch.tensor(random.uniform(-1,1), requires_grad=True) for _ in range(nin)]
        self.b = torch.tensor(random.uniform(-1.0,1.0), requires_grad=True)

    def parameters(self):
        return self.w

    def __call__(self, x):
        act = sum((wi*xi for wi,xi in zip(self.w, x)), self.b)
        return torch.tanh(act)

class Layer:

    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

    def __repr__(self):
        return f"Layer of [{', '.join(str(n) for n in self.neurons)}]"

class MLP:

    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

    def __repr__(self):
        return f"MLP of [{', '.join(str(layer) for layer in self.layers)}]"


In [495]:
# [weight(g), sweetness(1-10)] → fruit
x = [
    [150, 8],   # apple
    [120, 9],   # apple  
    [180, 7],   # apple
    [300, 3],   # orange
    [280, 4],   # orange
    [320, 2],   # orange
    [500, 6],   # watermelon
    [600, 7],   # watermelon
    [550, 5],   # watermelon
]
y = [0, 0, 0, 1, 1, 1, 2, 2, 2] 

xs = torch.tensor(x).float()
ys = torch.tensor(y)

n = MLP(2,[4,3])

In [496]:
# Loss function
def softmax(outputs):
    ex = [val.exp() for val in outputs]
    s = sum(ex)
    return [val/s for val in ex]
xs[:, 0] = xs[:, 0] / 600 
xs[:, 1] = xs[:, 1] / 10   

for i in range(10000):

    for w in n.parameters():
        w.grad = None
    
    loss = 0
    for j in range(len(xs)):
        probs = softmax(n(xs[j]))
        correct_class = ys[j].item()
        loss += -probs[correct_class].log()
    loss = loss / len(xs)
    

    loss.backward()
    for w in n.parameters():
        w.data -= 2 * w.grad
    
    if i % 100 == 0:
        print(f"step {i} | loss: {loss.item():.4f}")
    

step 0 | loss: 1.1293
step 100 | loss: 0.3751
step 200 | loss: 0.2751
step 300 | loss: 0.2658
step 400 | loss: 0.2585
step 500 | loss: 0.2548
step 600 | loss: 0.2523
step 700 | loss: 0.2506
step 800 | loss: 0.2493
step 900 | loss: 0.2483
step 1000 | loss: 0.2475
step 1100 | loss: 0.2469
step 1200 | loss: 0.2463
step 1300 | loss: 0.2458
step 1400 | loss: 0.2454
step 1500 | loss: 0.2450
step 1600 | loss: 0.2447
step 1700 | loss: 0.2444
step 1800 | loss: 0.2442
step 1900 | loss: 0.2440
step 2000 | loss: 0.2438
step 2100 | loss: 0.2436
step 2200 | loss: 0.2434
step 2300 | loss: 0.2432
step 2400 | loss: 0.2431
step 2500 | loss: 0.2430
step 2600 | loss: 0.2428
step 2700 | loss: 0.2427
step 2800 | loss: 0.2426
step 2900 | loss: 0.2425
step 3000 | loss: 0.2424
step 3100 | loss: 0.2423
step 3200 | loss: 0.2423
step 3300 | loss: 0.2422
step 3400 | loss: 0.2421
step 3500 | loss: 0.2420
step 3600 | loss: 0.2420
step 3700 | loss: 0.2419
step 3800 | loss: 0.2418
step 3900 | loss: 0.2418
step 4000 | 

In [489]:
def predict(x):
    x = torch.tensor(x, dtype=torch.float32)
    x[0] /= 600  
    x[1] /= 10
    probs = softmax(n(x))
    probs = [p.item() for p in probs]
    classes = ['apple', 'orange', 'watermelon']
    predicted = classes[probs.index(max(probs))]
    print(f"probs: {[f'{p:.2f}' for p in probs]}")
    print(f"predicted: {predicted}")

predict([160, 8])   
predict([290, 3])   
predict([520, 6])  

probs: ['0.79', '0.11', '0.11']
predicted: apple
probs: ['0.11', '0.79', '0.11']
predicted: orange
probs: ['0.11', '0.11', '0.79']
predicted: watermelon
